# LiH FIG.13 — find noiseless initial guess (seed sweep)

Exact **noiseless** VQE only: statevector `⟨H⟩`, parameter-shift gradients, Adam updates.
No gate depolarizing, shots, readout noise, CDR, or ZNE.

Scans **50** random initial-parameter seeds, keeps trajectories with
`err₀ < 10⁻¹` and `err₁₅ < 10⁻²` (vs exact ground-state energy), and plots the best final-error match.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import cirq
import matplotlib.pyplot as plt
import numpy as np
import sympy
from cirq.ops import GlobalPhaseGate

setattr(cirq, "GlobalPhaseGate", GlobalPhaseGate)

# --- repo paths ---
_repo = Path.cwd().resolve()
if not (_repo / "main_cursor_lib.py").is_file():
    _repo = _repo.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))


def lih_fig13_circuit(
    theta1: sympy.Symbol, theta2: sympy.Symbol, theta3: sympy.Symbol
) -> tuple[cirq.Circuit, list[cirq.LineQubit]]:
    q = cirq.LineQubit.range(6)
    q0, q1, q2, q3, q4, q5 = q
    c = cirq.Circuit()
    c.append(cirq.X(q0))
    c.append(cirq.X(q3))
    c.append(cirq.rx(np.pi / 2).on(q0))
    c.append(cirq.H(q2))
    c.append(cirq.H(q3))
    c.append(cirq.H(q4))
    c.append(cirq.H(q5))
    c.append([cirq.CZ(q0, q1), cirq.CZ(q3, q4)])
    c.append([cirq.CZ(q4, q5), cirq.H(q4)])
    c.append(cirq.CZ(q1, q4))
    c.append(cirq.rx(theta1).on(q1))
    c.append(cirq.CZ(q1, q4))
    c.append(cirq.CZ(q0, q1))
    c.append(cirq.H(q1))
    c.append(cirq.CZ(q0, q1))
    c.append(cirq.CZ(q1, q2))
    c.append(cirq.CZ(q1, q4))
    c.append(cirq.rx(theta2).on(q1))
    c.append(cirq.CZ(q1, q4))
    c.append(cirq.H(q4))
    c.append(cirq.CZ(q4, q5))
    c.append(cirq.H(q5))
    c.append(cirq.CZ(q3, q4))
    c.append(cirq.H(q4))
    c.append(cirq.CZ(q3, q4))
    c.append(cirq.H(q4))
    c.append(cirq.CZ(q1, q4))
    c.append(cirq.rx(theta3).on(q1))
    c.append(cirq.CZ(q1, q4))
    c.append(cirq.H(q4))
    c.append(cirq.CZ(q1, q2))
    c.append(cirq.H(q2))
    c.append(cirq.CZ(q0, q1))
    c.append(cirq.CZ(q3, q4))
    c.append(cirq.rx(-np.pi / 2).on(q0))
    c.append([cirq.H(q1), cirq.H(q3)])
    return c, q


def load_pauli_sum_from_numbered_file(path: Path, qubits: list[cirq.Qid]) -> cirq.PauliSum:
    idx_to_pauli = {1: cirq.X, 2: cirq.Y, 3: cirq.Z}
    out = cirq.PauliSum()
    with path.open("r", encoding="utf-8") as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.strip()
            if not line:
                continue
            parts = line.split()
            coeff = float(parts[0])
            pauli_codes = [int(x) for x in parts[1:]]
            if len(pauli_codes) != len(qubits):
                raise ValueError(f"{path}:{lineno}: expected {len(qubits)} Pauli indices.")
            pauli_string = cirq.PauliString()
            for q, code in zip(qubits, pauli_codes):
                if code == 0:
                    continue
                pauli_string *= idx_to_pauli[code](q)
            out += coeff * pauli_string
    return out


theta1 = sympy.Symbol("theta1")
theta2 = sympy.Symbol("theta2")
theta3 = sympy.Symbol("theta3")
circuit, qubits = lih_fig13_circuit(theta1, theta2, theta3)
qubit_map = {q: i for i, q in enumerate(qubits)}

bond_length = 2.2
ham_path = _repo / "Pauli_Ham" / f"LiH_bond_{bond_length:.1f}.txt"
pauli_sum = load_pauli_sum_from_numbered_file(ham_path, list(qubits))

hmat = pauli_sum.matrix(qubits=qubits)
e_gs = float(np.linalg.eigvalsh(hmat)[0].real)

print(f"LiH bond {bond_length} Å  |  Hamiltonian: {ham_path.name}")
print(f"Ground-state energy e_gs = {e_gs:.12f} Eh")

## Exact noiseless VQE + 50-seed sweep

Each seed draws $\theta_0 \sim \mathrm{Uniform}[-0.5, 0.5]^3$, then runs **15** Adam steps
(parameter-shift, LR = 0.03) on the exact statevector objective.

In [ ]:
# --- exact noiseless objective + Adam VQE ---

NUM_SEEDS = 50
SEED_BASE = 0
VQE_ITERS = 15
VQE_LR = 0.03
VQE_ADAM_BETA1 = 0.9
VQE_ADAM_BETA2 = 0.999
VQE_ADAM_EPS = 1e-8
INIT_LOW, INIT_HIGH = -0.5, 0.5
PARAM_SHIFT = np.pi / 2.0

ERR_START_MAX = 1e-1
ERR_END_MAX = 1e-2
ERR_FLOOR = 1e-16

_sim = cirq.Simulator()


def _resolver_from_params(params_vec: np.ndarray) -> cirq.ParamResolver:
    p = np.asarray(params_vec, dtype=float).reshape(3)
    return cirq.ParamResolver({theta1: float(p[0]), theta2: float(p[1]), theta3: float(p[2])})


def noiseless_energy(params_vec: np.ndarray) -> float:
    resolved = cirq.resolve_parameters(circuit, _resolver_from_params(params_vec))
    psi = np.asarray(_sim.simulate(resolved, qubit_order=qubits).final_state_vector, dtype=np.complex128)
    return float(np.real(pauli_sum.expectation_from_state_vector(psi, qubit_map=qubit_map)))


def parameter_shift_gradient(params_vec: np.ndarray) -> np.ndarray:
    g = np.zeros(3, dtype=float)
    p = np.asarray(params_vec, dtype=float).reshape(3)
    for i in range(3):
        plus, minus = p.copy(), p.copy()
        plus[i] += PARAM_SHIFT
        minus[i] -= PARAM_SHIFT
        g[i] = 0.5 * (noiseless_energy(plus) - noiseless_energy(minus))
    return g


def adam_apply(theta, grad, m, v, t, lr):
    t = int(t) + 1
    m = VQE_ADAM_BETA1 * m + (1.0 - VQE_ADAM_BETA1) * grad
    v = VQE_ADAM_BETA2 * v + (1.0 - VQE_ADAM_BETA2) * (grad * grad)
    mhat = m / (1.0 - VQE_ADAM_BETA1**t)
    vhat = v / (1.0 - VQE_ADAM_BETA2**t)
    return theta - lr * mhat / (np.sqrt(vhat) + VQE_ADAM_EPS), m, v, t


def run_noiseless_vqe(seed: int) -> dict:
    rng = np.random.default_rng(int(seed))
    params0 = rng.uniform(INIT_LOW, INIT_HIGH, size=3).astype(float)
    theta = params0.copy()
    m = np.zeros(3, dtype=float)
    v = np.zeros(3, dtype=float)
    t = 0

    energies = [noiseless_energy(theta)]
    for _ in range(int(VQE_ITERS)):
        grad = parameter_shift_gradient(theta)
        theta, m, v, t = adam_apply(theta, grad, m, v, t, float(VQE_LR))
        energies.append(noiseless_energy(theta))

    energies = np.asarray(energies, dtype=float)
    errors = np.maximum(np.abs(energies - e_gs), ERR_FLOOR)
    return {
        "seed": int(seed),
        "params0": params0,
        "energies": energies,
        "errors": errors,
        "err_start": float(errors[0]),
        "err_end": float(errors[-1]),
    }


seeds = list(range(SEED_BASE, SEED_BASE + NUM_SEEDS))
results = [run_noiseless_vqe(s) for s in seeds]

qualified = [
    r
    for r in results
    if r["err_start"] < ERR_START_MAX and r["err_end"] < ERR_END_MAX
]
if qualified:
    selected = min(qualified, key=lambda r: r["err_end"])
    selection_note = f"{len(qualified)}/{NUM_SEEDS} seeds met thresholds; picked lowest final error."
else:
    selected = min(results, key=lambda r: r["err_end"])
    selection_note = (
        f"No seed met err₀<{ERR_START_MAX:g} and err₁₅<{ERR_END_MAX:g}; "
        f"showing best final error among all {NUM_SEEDS} seeds."
    )

print(selection_note)
print(f"Selected seed: {selected['seed']}")
print(f"  err_start = {selected['err_start']:.6e}  (target < {ERR_START_MAX:g})")
print(f"  err_end   = {selected['err_end']:.6e}  (target < {ERR_END_MAX:g})")
print(f"  E_start   = {selected['energies'][0]:.10f} Eh")
print(f"  E_end     = {selected['energies'][-1]:.10f} Eh")
print(f"  constraints_met = {selected['err_start'] < ERR_START_MAX and selected['err_end'] < ERR_END_MAX}")

In [ ]:
# --- plot selected trajectory ---

xs = np.arange(len(selected["errors"]), dtype=int)
errs = selected["errors"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(xs, errs, color="#2ca02c", marker="*", ms=8, lw=1.5, label=f"seed {selected['seed']}")
ax.axhline(ERR_END_MAX, color="#6dd5ed", ls="--", lw=1.2, alpha=0.8, label=f"target end < {ERR_END_MAX:g}")
ax.axhline(ERR_START_MAX, color="#aaaaaa", ls=":", lw=1.0, alpha=0.8, label=f"target start < {ERR_START_MAX:g}")
ax.set_xlabel("Iteration")
ax.set_ylabel(r"$|E - E_{\mathrm{gs}}|$")
ax.set_title("Exact noiseless VQE — selected initial-guess seed")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="best")
plt.tight_layout()
plt.show()